# TARDIS — Data Cleaning

This notebook cleans and prepares the **SNCF TGV punctuality dataset** for downstream modelling.

The pipeline runs in order — each step depends on the previous one:

| # | Step | What it does |
|---|------|-------------|
| 2 | Load | Read raw CSV, snapshot the original |
| 2.1 | Drop comment columns | Remove free-text columns unusable in a tabular model |
| 3 | Deduplicate | Remove exact duplicate rows |
| 4 | Map missing values | Audit NaN extent before any action |
| 4.1 | Drop critical NaN | Drop rows missing an identifier that cannot be recovered |
| 4.2–4.5 | Standardise types | Parse dates, convert numerics, cast text, normalise labels |
| 4.5.1 | Station name clustering | Fuzzy deduplication of station name variants |
| 4.6–4.7 | Recover delay NaN | Derive missing delay values from algebraic relationships |
| 4.8 | Report remaining NaN | Document intentionally kept nulls |
| 5 | Cleaning summary | Shape delta: raw vs cleaned + retention funnel |
| 6–10 | Feature engineering | Year, month, season, delay category, cancellation/punctuality rates |
| 11–11.3 | Correct impossible values | Negatives, count overflows, delay hierarchy violations |
| 12 | Export dataset | Write `../cleaned_dataset.csv` |
| 13 | Data quality audit | Completeness, distributions, outliers, validity checks, effectiveness |
| 14 | Export audit report | Write `../cleaning_report.csv` |

### 1. Imports

In [100]:
import re
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

> **Tracking** — initialise the audit report accumulator. Each step appends entries to this list; the full report is written to CSV at step 14.

In [101]:
report = []

### 2. Data loading

Raw CSV uses `;` as separator. We keep an untouched copy (`original_file`) to compute row/column deltas at the end and to populate the audit report.

In [102]:
df = pd.read_csv("./dataset.csv", sep=";")
original_file = df.copy()
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')

Loaded: 12070 rows, 26 columns


> **Tracking** — snapshot the raw dataset: total shape, total null count, and per-column null counts. These become the `initial_state` entries in the audit report.

In [103]:
_initial_null_total = original_file.isnull().sum().sum()
_perfect_rows       = original_file.dropna().shape[0]
_perfect_cols       = original_file.dropna(axis=1).shape[1]

report += [
    {'metric': 'initial_rows',               'value': original_file.shape[0],                 'category': 'initial_state', 'reason': 'Raw dataset before any cleaning'},
    {'metric': 'initial_columns',            'value': original_file.shape[1],                 'category': 'initial_state', 'reason': 'Raw dataset before any cleaning'},
    {'metric': 'initial_null_total',         'value': int(_initial_null_total),               'category': 'initial_state', 'reason': 'Total missing values across all cells'},
    {'metric': 'perfect_rows',               'value': _perfect_rows,                          'category': 'initial_state', 'reason': 'Rows with zero NaN — no cleaning needed'},
    {'metric': 'rows_lost_if_strict_dropna', 'value': original_file.shape[0] - _perfect_rows, 'category': 'initial_state', 'reason': 'Rows lost with a strict zero-NaN policy'},
    {'metric': 'perfect_columns',            'value': _perfect_cols,                          'category': 'initial_state', 'reason': 'Columns with zero NaN'},
    {'metric': 'cols_lost_if_strict_dropna', 'value': original_file.shape[1] - _perfect_cols, 'category': 'initial_state', 'reason': 'Columns lost with a strict zero-NaN policy'},
]
for col in original_file.columns:
    report.append({'metric': f'initial_null__{col}', 'value': int(original_file[col].isnull().sum()), 'category': 'initial_null_per_column', 'reason': col})

print(f'Initial null total : {_initial_null_total}')
print(f'Perfect rows       : {_perfect_rows} / {original_file.shape[0]}')
print(f'Perfect columns    : {_perfect_cols} / {original_file.shape[1]}')

Initial null total : 38874
Perfect rows       : 2 / 12070
Perfect columns    : 0 / 26


### 2.1 Dropping comment columns

Columns whose name ends with `comments` contain free-text justifications written by operators. They cannot be encoded or aggregated in a tabular model, so we drop them entirely.

In [104]:
comment_cols = df.filter(regex="comments$").columns.tolist()
df = df.drop(columns=comment_cols)
print(f'Dropped {len(comment_cols)} column(s): {comment_cols}')
print(f'Remaining: {df.shape[1]} columns')

Dropped 3 column(s): ['Cancellation comments', 'Departure delay comments', 'Arrival delay comments']
Remaining: 23 columns


> **Tracking** — log each dropped column by name.

In [105]:
for col in comment_cols:
    report.append({'metric': 'column_dropped', 'value': col, 'category': 'cleaning', 'reason': 'Irrelevant data — free text comment column'})
report.append({'metric': 'columns_dropped_total', 'value': len(comment_cols), 'category': 'cleaning', 'reason': 'Irrelevant data — free text columns removed'})
print(f'Tracked: {len(comment_cols)} column(s) logged.')

Tracked: 3 column(s) logged.


### 3. Duplicate check

Duplicate rows artificially inflate all aggregated statistics (means, totals, distributions). We check and remove exact duplicates before any other cleaning step.

In [106]:
nb_dup = df.duplicated().sum()
print(f'Duplicates found: {nb_dup}')
if nb_dup > 0:
    df = df.drop_duplicates()
    print(f'Removed. \nRows remaining: {len(df)}')
else:
    print('No duplicates found.')

Duplicates found: 174
Removed. 
Rows remaining: 11896


> **Tracking** — log the number of duplicate rows removed.

In [107]:
report.append({'metric': 'rows_dropped_duplicates', 'value': int(nb_dup), 'category': 'cleaning', 'reason': 'Duplicate rows — same observation counted more than once'})
print(f'Tracked: {nb_dup} row(s) logged.')

Tracked: 174 row(s) logged.


### 4. Missing values — overview

Before taking any action, we map the full extent of missing data. This guides the strategy for each column: **drop the row** (step 4.1), **derive the value** from existing row data (steps 4.6–4.7), or **leave it null** intentionally (step 4.8).

In [108]:
nan_counts = df.isnull().sum()
print('Missing values per column:')
print(nan_counts[nan_counts > 0].to_string())
print(f'\nTotal: {nan_counts.sum()}')

Missing values per column:
Date                                                                              59
Service                                                                          240
Departure station                                                                 59
Arrival station                                                                   59
Average journey time                                                             236
Number of scheduled trains                                                       237
Number of cancelled trains                                                       237
Number of trains delayed at departure                                            238
Average delay of late trains at departure                                        238
Average delay of all trains at departure                                         237
Number of trains delayed at arrival                                              237
Average delay of late trains at arriva

### 4.1 Dropping rows with missing critical identifiers

A row is dropped only when one of these five columns is null:
`Date`, `Service`, `Departure station`, `Arrival station`, `Number of scheduled trains`.

Without these, a row cannot be placed in time, attributed to a route, or used to compute any rate — it is unrecoverable. All other NaN values are handled later.

In [109]:
before = len(df)
df = df.dropna(subset=["Date", "Service", "Departure station", "Arrival station", "Number of scheduled trains"])
print(f'Rows dropped   : {before - len(df)}')
print(f'Rows remaining : {len(df)}')

Rows dropped   : 636
Rows remaining : 11260


> **Tracking** — log the number of rows removed due to missing critical identifiers.

In [110]:
_crit_dropped = before - len(df)
report.append({'metric': 'rows_dropped_critical_nan', 'value': _crit_dropped, 'category': 'cleaning', 'reason': 'Unusable for analysis — missing key identifiers (CRITICAL_COLS)'})
print(f'Tracked: {_crit_dropped} row(s) logged.')

Tracked: 636 row(s) logged.


### 4.2 Date parsing

The `Date` column contains mixed string formats across years. `format='mixed'` lets pandas infer the format row by row, which is more robust than a fixed strftime pattern. The column is cast to `datetime64`.

In [111]:
df["Date"] = pd.to_datetime(df["Date"], format='mixed')
print(f'dtype : {df["Date"].dtype}')
print(f'Range : {df["Date"].min().date()} → {df["Date"].max().date()}')

dtype : datetime64[ns]
Range : 2018-01-01 → 2025-12-01


### 4.3 Numeric conversion

All columns except `Date`, `Service`, `Departure station`, and `Arrival station` should be numeric. Some cells use a comma as decimal separator — those are normalised to a dot before conversion. The dtype guard prevents calling `.str` on columns that are already numeric, which would silently produce NaN.

In [112]:
text_cols = ["Date", "Service", "Departure station", "Arrival station"]
numeric_cols = [c for c in df.columns if c not in text_cols]

for col in numeric_cols:
    if df[col].dtype == object:
        df[col] = df[col].str.strip().str.replace(',', '.')
    df[col] = pd.to_numeric(df[col], errors='coerce')

still_object = [c for c in numeric_cols if df[c].dtype == object]
if still_object:
    print(f'WARNING — still object after conversion: {still_object}')
else:
    print(f'All {len(numeric_cols)} numeric columns successfully converted.')

All 19 numeric columns successfully converted.


### 4.4 String dtype cast

By default, pandas reads text columns as `object`, which allows mixed types (strings, `None`, floats). Casting to `string` enforces a uniform dtype and makes null handling explicit (`pd.NA` instead of `None`).

In [113]:
str_cols = ["Service", "Departure station", "Arrival station"]
df[str_cols] = df[str_cols].astype('string')
print('Dtypes after cast:')
print(df[str_cols].dtypes.to_string())

Dtypes after cast:
Service              string[python]
Departure station    string[python]
Arrival station      string[python]


### 4.5 Label normalisation

Station and service names can appear with inconsistent casing or surrounding spaces (e.g. `"paris montparnasse"` vs `"PARIS MONTPARNASSE "`). Stripping and uppercasing ensures these are treated as the same entity in groupby operations.

In [114]:
label_cols = ["Departure station", "Arrival station", "Service"]
for col in label_cols:
    df[col] = df[col].str.strip().str.upper()

for col in label_cols:
    print(f'Unique {col}: {df[col].nunique()}')

Unique Departure station: 64
Unique Arrival station: 63
Unique Service: 2


### 4.5.1 Station name clustering

Fuzzy station-name canonicalization using Union-Find and `token_sort_ratio`. Names that match at or above `STATION_THRESHOLD` are considered the same station and merged into a single canonical spelling chosen from observed data.

**Similarity threshold and sentinel values.** `STATION_THRESHOLD` controls cluster sensitivity: two names with a `token_sort_ratio` score >= 80 are treated as equivalent. `_CONFUSABLES` normalizes common visual substitutions (for example `1` -> `I`, `3` -> `E`) before matching. `_INVALID` lists placeholders and null-like strings that do not represent real station names and must be excluded.

In [115]:
_CONFUSABLES = str.maketrans({
    "1": "I", "3": "E", "4": "A", "5": "S", "8": "B",
    "@": "A", "'": "", "`": "", "-": ""
})
_ABBREV = {"ST ": "SAINT "}
_INVALID = {"", "0", "NAN", "NONE", "NULL", "N/A", "NA"}
STATION_THRESHOLD = 80

**Preprocessing and station-column detection (inline).** Names are standardized by uppercasing, collapsing spaces, applying `_CONFUSABLES`, and expanding abbreviations (`ST` -> `SAINT`). Columns are kept only if they match station-related keywords, are string-typed, and satisfy the station-like heuristic (short labels, low digit ratio, low cardinality).

In [116]:
keywords = ("station", "departure", "arrival", "gare", "origine", "destination")
_station_columns = []

for col in df.columns:
    if not any(kw in col.lower() for kw in keywords):
        continue
    if not pd.api.types.is_string_dtype(df[col]):
        continue

    sample = df[col].dropna().astype(str)
    if sample.empty:
        continue

    avg_len = sample.str.len().mean()
    digit_ratio = sample.str.count(r"\d").sum() / max(sample.str.len().sum(), 1)
    unique_ratio = df[col].nunique() / max(len(df[col]), 1)

    if avg_len < 50 and digit_ratio < 0.05 and unique_ratio < 0.1 and df[col].nunique() >= 5:
        _station_columns.append(col)

print(f'Station-like columns detected: {_station_columns}')

Station-like columns detected: ['Departure station', 'Arrival station']


**Inline Union-Find clustering (no helper functions).** All station values are pre-cleaned first. Invalid placeholders are filtered out before pairwise fuzzy matching. Cluster unions use path compression and frequency-based root attachment so common spellings dominate as canonical representatives.

In [117]:
_all_values = []
for col in _station_columns:
    _all_values.extend(df[col].astype(str).tolist())

_cleaned_of_raw = {}
_valid_cleaned = []

for raw in _all_values:
    cleaned = str(raw).strip().upper().translate(_CONFUSABLES)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    for abbr, full in _ABBREV.items():
        cleaned = cleaned.replace(abbr, full)

    _cleaned_of_raw[raw] = cleaned
    if cleaned not in _INVALID:
        _valid_cleaned.append(cleaned)

counts = Counter(_valid_cleaned)
unique = list(counts.keys())
parent = {v: v for v in unique}

def _tok_sort_ratio(a, b):
    a_s = " ".join(sorted(a.split()))
    b_s = " ".join(sorted(b.split()))
    t = len(a_s) + len(b_s)
    if t == 0:
        return 100
    common = sum((Counter(a_s) & Counter(b_s)).values())
    return int(200 * common / t)

for i, a in enumerate(unique):
    for b in unique[i + 1:]:
        if _tok_sort_ratio(a, b) < STATION_THRESHOLD:
            continue

        ra = a
        while parent[ra] != ra:
            parent[ra] = parent[parent[ra]]
            ra = parent[ra]

        rb = b
        while parent[rb] != rb:
            parent[rb] = parent[parent[rb]]
            rb = parent[rb]

        if ra == rb:
            continue
        if counts[ra] >= counts[rb]:
            parent[rb] = ra
        else:
            parent[ra] = rb

**Build and apply the canonical mapping.** Cluster roots are resolved with path compression, each cluster gets its most frequent label as canonical form, then every raw value is mapped to that canonical label and written back to the detected station columns.

In [118]:
_root_to_members = {}
for v in unique:
    r = v
    while parent[r] != r:
        parent[r] = parent[parent[r]]
        r = parent[r]
    _root_to_members.setdefault(r, []).append(v)

_canonical_of = {}
for members in _root_to_members.values():
    canonical = max(members, key=lambda x: counts[x])
    for member in members:
        _canonical_of[member] = canonical

_mapping = {}
for raw in _all_values:
    _mapping[raw] = _canonical_of.get(_cleaned_of_raw.get(raw))

for col in _station_columns:
    df[col] = df[col].astype(str).map(_mapping)

**Cluster verification output.** For each canonical label that absorbs multiple variants, the notebook prints all merged raw spellings. Single-member clusters are intentionally omitted, so the output highlights only real merges and makes manual validation of `STATION_THRESHOLD` calibration straightforward.

In [119]:
_clusters: dict = {}
for raw, canon in _mapping.items():
    _clusters.setdefault(canon, []).append(raw)

for canon, variants in sorted(_clusters.items(), key=lambda x: str(x[0])):
    if len(variants) > 1:
        print(f"\n✦ {canon}")
        for v in sorted(variants, key=str):
            print(f"    {'  ' if v == canon else '↳ '}{v!r}")


✦ ANGERS SAINT LAUD
      'ANGERS SAINT LAUD'
    ↳ 'ANGERS ST LAUD'

✦ BORDEAUX SAINT JEAN
      'BORDEAUX SAINT JEAN'
    ↳ 'BORDEAUX ST JEAN'

✦ MARSEILLE SAINT CHARLES
      'MARSEILLE SAINT CHARLES'
    ↳ 'MARSEILLE ST CHARLES'

✦ NANCY
    ↳ 'ANNECY'
      'NANCY'

✦ NANTES
      'NANTES'
    ↳ 'VANNES'

✦ PARIS LYON
      'PARIS LYON'
    ↳ 'PARIS NORD'

✦ REIMS
    ↳ 'NIMES'
      'REIMS'

✦ SAINT ETIENNE CHATEAUCREUX
      'SAINT ETIENNE CHATEAUCREUX'
    ↳ 'ST ETIENNE CHATEAUCREUX'

✦ SAINT MALO
      'SAINT MALO'
    ↳ 'ST MALO'


> **Tracking** — station name fuzzy clustering statistics: cluster count, merge count, and number of variant spellings unified.

In [120]:
_n_clusters  = len(_root_to_members)
_n_merged    = sum(1 for m in _root_to_members.values() if len(m) > 1)
_n_variants  = sum(len(m) - 1 for m in _root_to_members.values() if len(m) > 1)

report += [
    {'metric': 'station_clusters_total',     'value': _n_clusters,           'category': 'station_clustering', 'reason': 'Unique canonical station names after fuzzy deduplication'},
    {'metric': 'station_clusters_with_merge','value': _n_merged,             'category': 'station_clustering', 'reason': 'Canonical names that absorbed at least one spelling variant'},
    {'metric': 'station_variants_merged',    'value': _n_variants,           'category': 'station_clustering', 'reason': 'Non-canonical spellings unified into a canonical form'},
    {'metric': 'station_fuzzy_threshold',    'value': STATION_THRESHOLD,     'category': 'station_clustering', 'reason': 'token_sort_ratio threshold used for fuzzy matching'},
    {'metric': 'station_columns_processed',  'value': len(_station_columns), 'category': 'station_clustering', 'reason': 'Number of station-type columns processed'},
]
print(f'Clusters total    : {_n_clusters}')
print(f'Clusters merged   : {_n_merged}')
print(f'Variants merged   : {_n_variants}')
print(f'Threshold used    : {STATION_THRESHOLD}')

Clusters total    : 55
Clusters merged   : 4
Variants merged   : 4
Threshold used    : 80


### 4.6 Recovering departure delay NaN

The two departure-delay metrics are algebraically linked. When one is missing but the counterpart and required train counts are available, the missing value is computed exactly (deterministic recovery, not statistical imputation):

- `dep_late = dep_all × (scheduled / n_delayed)`  ← when `dep_late` is null
- `dep_all  = dep_late × (n_delayed / scheduled)`  ← when `dep_all` is null

In [121]:
col_dep_late = "Average delay of late trains at departure"
col_dep_all  = "Average delay of all trains at departure"
col_n_dep    = "Number of trains delayed at departure"

mask = df[col_dep_late].isna() & df[col_dep_all].notna() & (df[col_n_dep] > 0)
_dep_late_filled = int(mask.sum())
df.loc[mask, col_dep_late] = df.loc[mask, col_dep_all] * df.loc[mask, "Number of scheduled trains"] / df.loc[mask, col_n_dep]
print(f'dep_late filled from dep_all : {_dep_late_filled} row(s)')

mask = df[col_dep_all].isna() & df[col_dep_late].notna() & df[col_n_dep].notna()
_dep_all_filled = int(mask.sum())
df.loc[mask, col_dep_all] = df.loc[mask, col_dep_late] * df.loc[mask, col_n_dep] / df.loc[mask, "Number of scheduled trains"]
print(f'dep_all  filled from dep_late : {_dep_all_filled} row(s)')

dep_late filled from dep_all : 269 row(s)
dep_all  filled from dep_late : 279 row(s)


> **Tracking** — log how many cells were recovered for each departure delay column.

In [122]:
report += [
    {'metric': 'values_filled_dep_late', 'value': _dep_late_filled, 'category': 'data_recovery', 'reason': 'Computed from average departure delay + scheduled/delayed counts'},
    {'metric': 'values_filled_dep_all',  'value': _dep_all_filled,  'category': 'data_recovery', 'reason': 'Computed from late-train departure delay + delayed/scheduled counts'},
]
print(f'Tracked: dep_late={_dep_late_filled}, dep_all={_dep_all_filled}')

Tracked: dep_late=269, dep_all=279


### 4.7 Recovering arrival delay NaN

Same deterministic algebraic recovery as step 4.6, applied to arrival-delay metrics.

In [123]:
col_arr_late = "Average delay of late trains at arrival"
col_arr_all  = "Average delay of all trains at arrival"
col_n_arr    = "Number of trains delayed at arrival"

mask = df[col_arr_late].isna() & df[col_arr_all].notna() & (df[col_n_arr] > 0)
_arr_late_filled = int(mask.sum())
df.loc[mask, col_arr_late] = df.loc[mask, col_arr_all] * df.loc[mask, "Number of scheduled trains"] / df.loc[mask, col_n_arr]
print(f'arr_late filled from arr_all : {_arr_late_filled} row(s)')

mask = df[col_arr_all].isna() & df[col_arr_late].notna() & df[col_n_arr].notna()
_arr_all_filled = int(mask.sum())
df.loc[mask, col_arr_all] = df.loc[mask, col_arr_late] * df.loc[mask, col_n_arr] / df.loc[mask, "Number of scheduled trains"]
print(f'arr_all  filled from arr_late : {_arr_all_filled} row(s)')

arr_late filled from arr_all : 266 row(s)
arr_all  filled from arr_late : 266 row(s)


> **Tracking** — log how many cells were recovered for each arrival delay column.

In [124]:
report += [
    {'metric': 'values_filled_arr_late', 'value': _arr_late_filled, 'category': 'data_recovery', 'reason': 'Computed from average arrival delay + scheduled/delayed counts'},
    {'metric': 'values_filled_arr_all',  'value': _arr_all_filled,  'category': 'data_recovery', 'reason': 'Computed from late-train arrival delay + delayed/scheduled counts'},
]
print(f'Tracked: arr_late={_arr_late_filled}, arr_all={_arr_all_filled}')

Tracked: arr_late=266, arr_all=266


### 4.8 Remaining NaN — status

Any NaN still present after steps 4.6–4.7 cannot be derived from existing row data. These are left null intentionally — imputing them would introduce artificial signal. They are preserved as-is in the exported dataset.

In [125]:
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]
if not remaining.empty:
    print('Columns with remaining NaN (left as null intentionally):')
    print(remaining.to_string())
else:
    print('No remaining NaN.')

Columns with remaining NaN (left as null intentionally):
Departure station                                                                  4
Arrival station                                                                   11
Average journey time                                                             286
Number of cancelled trains                                                       223
Number of trains delayed at departure                                            228
Average delay of late trains at departure                                         10
Average delay of all trains at departure                                          10
Number of trains delayed at arrival                                              230
Average delay of late trains at arrival                                           18
Average delay of all trains at arrival                                             9
Number of trains delayed > 15min                                                 222
Average 

> **Tracking** — null count remaining after algebraic recovery and algebraic coverage rate (recovered / initial nulls).

In [126]:
_null_post_recovery      = int(df.isnull().sum().sum())
_initial_null_lookup     = next(r['value'] for r in report if r['metric'] == 'initial_null_total')
_algebraic_total         = sum(
    next(r['value'] for r in report if r['metric'] == m)
    for m in ['values_filled_dep_late', 'values_filled_dep_all', 'values_filled_arr_late', 'values_filled_arr_all']
)
_algebraic_recovery_rate = round(_algebraic_total / max(_initial_null_lookup, 1), 4)

report += [
    {'metric': 'null_post_recovery',        'value': _null_post_recovery,      'category': 'recovery_summary', 'reason': 'Remaining nulls after algebraic recovery (steps 4.6–4.7)'},
    {'metric': 'null_recovered_algebraic',  'value': _algebraic_total,         'category': 'recovery_summary', 'reason': 'Total cells recovered via algebraic derivation'},
    {'metric': 'algebraic_recovery_rate',   'value': _algebraic_recovery_rate, 'category': 'recovery_summary', 'reason': 'Recovered / initial_null_total — algebraic coverage of initial nulls'},
]
print(f'Nulls post recovery   : {_null_post_recovery}')
print(f'Recovered algebraic   : {_algebraic_total}')
print(f'Algebraic coverage    : {_algebraic_recovery_rate:.1%} of initial nulls')

Nulls post recovery   : 3636
Recovered algebraic   : 1080
Algebraic coverage    : 2.8% of initial nulls


### 5. Cleaning summary

Shape comparison between the raw dataset and the cleaned dataset after all removal steps.

In [127]:
print(f'Original : {original_file.shape[0]} rows, {original_file.shape[1]} columns')
print(f'Cleaned  : {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Removed  : {original_file.shape[0] - df.shape[0]} rows, {original_file.shape[1] - df.shape[1]} columns')

Original : 12070 rows, 26 columns
Cleaned  : 11260 rows, 23 columns
Removed  : 810 rows, 3 columns


> **Tracking** — row retention rate and step-by-step retention funnel across the full pipeline.

In [128]:
_row_retention   = round(len(df) / original_file.shape[0], 4)
_rows_lost_dedup = next(r['value'] for r in report if r['metric'] == 'rows_dropped_duplicates')
_rows_lost_crit  = next(r['value'] for r in report if r['metric'] == 'rows_dropped_critical_nan')
_cols_after_drop = original_file.shape[1] - len(comment_cols)

report += [
    {'metric': 'row_retention_rate',         'value': _row_retention,  'category': 'pipeline_summary', 'reason': 'Fraction of original rows kept (higher = more data preserved)'},
    {'metric': 'pct_rows_lost_dedup',        'value': round(_rows_lost_dedup / original_file.shape[0], 4), 'category': 'pipeline_summary', 'reason': 'Fraction of rows lost to exact deduplication'},
    {'metric': 'pct_rows_lost_critical_nan', 'value': round(_rows_lost_crit  / original_file.shape[0], 4), 'category': 'pipeline_summary', 'reason': 'Fraction of rows lost to missing critical identifiers'},
    {'metric': 'cols_added_feature_eng',     'value': df.shape[1] - _cols_after_drop, 'category': 'pipeline_summary', 'reason': 'Net columns added by feature engineering (year, month, season, etc.)'},
]

_steps = [
    ('Raw dataset',                  original_file.shape[0]),
    ('After deduplication',          original_file.shape[0] - _rows_lost_dedup),
    ('After critical NaN drop',      original_file.shape[0] - _rows_lost_dedup - _rows_lost_crit),
    ('Final (cleaned)',              len(df)),
]
print(f'{"Step":<35} {"Rows":>8}  {"Retained":>9}')
print('─' * 56)
for step, n in _steps:
    print(f'{step:<35} {n:>8,}  {n / original_file.shape[0]:>8.1%}')

Step                                    Rows   Retained
────────────────────────────────────────────────────────
Raw dataset                           12,070    100.0%
After deduplication                   11,896     98.6%
After critical NaN drop               11,260     93.3%
Final (cleaned)                       11,260     93.3%


### 6. Year and month extraction

Extracted from the parsed `Date` column as separate integer columns. Used for temporal grouping, trend analysis, and seasonal encoding (step 7).

In [129]:
df["year"]  = df["Date"].dt.year
df["month"] = df["Date"].dt.month
print(f'Years  : {sorted(df["year"].unique().tolist())}')
print(f'Months : {sorted(df["month"].unique().tolist())}')

Years  : [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Months : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


### 7. Season encoding

Northern-hemisphere meteorological seasons mapped from the month integer. Season is a relevant feature: winter brings weather disruptions, summer increases passenger load — both affect delay patterns.

In [130]:
season_map = {
    12:'winter',1:'winter',2:'winter',
    3:'spring',4:'spring',5:'spring',
    6:'summer',7:'summer',8:'summer',
    9:'autumn',10:'autumn',11:'autumn',
}
df["season"] = df["month"].map(season_map)
print('Season distribution:')
print(df["season"].value_counts().to_string())

Season distribution:
season
summer    2832
autumn    2820
winter    2818
spring    2790


### 8. Delay severity category

Average arrival delay is binned into five ordered categories based on operational thresholds used by SNCF:

| Label | Delay range |
|-------|-------------|
| `early` | < 0 min (arrived early) |
| `on_time` | 0–5 min |
| `slight` | 5–15 min |
| `moderate` | 15–30 min |
| `severe` | > 30 min |

In [131]:
df["delay_category"] = pd.cut(
    df["Average delay of all trains at arrival"],
    bins=[float('-inf'), 0, 5, 15, 30, float('inf')],
    labels=['early', 'on_time', 'slight', 'moderate', 'severe']
)
print('Delay category distribution:')
print(df["delay_category"].value_counts().sort_index().to_string())

Delay category distribution:
delay_category
early        219
on_time     4951
slight      5693
moderate     373
severe        15


### 9. Cancellation rate

Proportion of scheduled trains that were cancelled: `cancelled / scheduled`.
Ranges from `0.0` (no cancellations) to `1.0` (all trains cancelled).

In [132]:
df["cancellation_rate"] = df["Number of cancelled trains"] / df["Number of scheduled trains"]
print(f'cancellation_rate — min: {df["cancellation_rate"].min():.4f}  mean: {df["cancellation_rate"].mean():.4f}  max: {df["cancellation_rate"].max():.4f}')

cancellation_rate — min: 0.0000  mean: inf  max: inf


### 10. Punctuality rate

Proportion of trains that arrived on time: `1 − (delayed_at_arrival / scheduled)`.
A value of `1.0` means no train was delayed; `0.0` means every train was delayed.

In [133]:
df["punctuality_rate"] = 1 - (df["Number of trains delayed at arrival"] / df["Number of scheduled trains"])
print(f'punctuality_rate — min: {df["punctuality_rate"].min():.4f}  mean: {df["punctuality_rate"].mean():.4f}  max: {df["punctuality_rate"].max():.4f}')

punctuality_rate — min: 0.0000  mean: 0.8610  max: 1.0000


### 11. Negative value correction

Train count columns cannot be negative by definition — a negative count is a data entry error. Negative values are replaced with the **median of valid (≥ 0) values for the same route** (`Departure station` × `Arrival station`). Using a route-level median preserves the local distribution rather than pulling the value towards a global average.

In [134]:
count_cols = [
    "Number of scheduled trains",
    "Number of cancelled trains",
    "Number of trains delayed at departure",
    "Number of trains delayed at arrival",
    "Number of trains delayed > 15min",
    "Number of trains delayed > 30min",
    "Number of trains delayed > 60min",
]
route_cols = ["Departure station", "Arrival station"]

_neg_fixed_total = 0
for col in count_cols:
    mask = df[col] < 0
    if mask.any():
        route_median = df.groupby(route_cols)[col].transform(lambda x: x[x >= 0].median())
        df.loc[mask, col] = route_median[mask]
        _neg_fixed_total += int(mask.sum())
        print(f'[FIXED] {col}: {mask.sum()} negative value(s) → route median')
print(f'Total: {_neg_fixed_total} fix(es)')

[FIXED] Number of trains delayed > 30min: 41 negative value(s) → route median
Total: 41 fix(es)


> **Tracking** — log the total number of negative count values corrected across all columns.

In [135]:
report.append({'metric': 'values_fixed_negative', 'value': _neg_fixed_total, 'category': 'data_correction', 'reason': 'Impossible negative count values — replaced by route-level median'})
print(f'Tracked: {_neg_fixed_total} fix(es) logged.')

Tracked: 41 fix(es) logged.


### 11.1 Consistency — counts vs scheduled trains

`Number of cancelled trains`, `Number of trains delayed at departure`, and `Number of trains delayed at arrival` cannot logically exceed the total number of scheduled trains on the same row. Violations are replaced by the route-level median of valid values.

In [136]:
counts_vs_scheduled = [
    "Number of cancelled trains",
    "Number of trains delayed at departure",
    "Number of trains delayed at arrival",
]

_cons_count_fixed = 0
for col in counts_vs_scheduled:
    mask = df[col].notna() & (df[col] > df["Number of scheduled trains"])
    if mask.any():
        route_median = df.groupby(route_cols)[col].transform(
            lambda x, c=col: x[x <= df.loc[x.index, "Number of scheduled trains"]].median()
        )
        df.loc[mask, col] = route_median[mask]
        _cons_count_fixed += int(mask.sum())
        print(f'[FIXED] {col}: {mask.sum()} value(s) exceeded scheduled → route median')
print(f'Total: {_cons_count_fixed} fix(es)')

[FIXED] Number of cancelled trains: 54 value(s) exceeded scheduled → route median
[FIXED] Number of trains delayed at departure: 15 value(s) exceeded scheduled → route median
Total: 69 fix(es)


> **Tracking** — log the number of values corrected for exceeding the scheduled train count.

In [137]:
report.append({'metric': 'values_fixed_count_vs_scheduled', 'value': _cons_count_fixed, 'category': 'data_correction', 'reason': 'Counts exceeding scheduled trains — logically impossible'})
print(f'Tracked: {_cons_count_fixed} fix(es) logged.')

Tracked: 69 fix(es) logged.


### 11.2 Consistency — delay hierarchy

By definition, every train delayed more than 30 min was also delayed more than 15 min, so the count hierarchy `>15min ≥ >30min ≥ >60min` must hold. Violations are corrected **bottom-up** (starting from the highest threshold) using the route-level median of valid values.

In [138]:
delay_hierarchy = [
    "Number of trains delayed > 15min",
    "Number of trains delayed > 30min",
    "Number of trains delayed > 60min",
]

_hier_fixed = 0
for i in range(len(delay_hierarchy) - 1):
    col_higher = delay_hierarchy[i + 1]
    col_lower  = delay_hierarchy[i]
    mask = df[col_higher].notna() & df[col_lower].notna() & (df[col_higher] > df[col_lower])
    if mask.any():
        route_median = df.groupby(route_cols)[col_higher].transform(
            lambda x, cl=col_lower: x[x <= df.loc[x.index, cl]].median()
        )
        df.loc[mask, col_higher] = route_median[mask]
        _hier_fixed += int(mask.sum())
        print(f'[FIXED] {col_higher} > {col_lower}: {mask.sum()} row(s)')
print(f'Total: {_hier_fixed} fix(es)')

[FIXED] Number of trains delayed > 30min > Number of trains delayed > 15min: 3 row(s)
[FIXED] Number of trains delayed > 60min > Number of trains delayed > 30min: 314 row(s)
Total: 317 fix(es)


> **Tracking** — log the number of delay hierarchy violations corrected.

In [139]:
report.append({'metric': 'values_fixed_delay_hierarchy', 'value': _hier_fixed, 'category': 'data_correction', 'reason': 'Delay hierarchy violation — higher threshold exceeded lower threshold'})
print(f'Tracked: {_hier_fixed} fix(es) logged.')

Tracked: 317 fix(es) logged.


### 11.3 Recomputing derived rates

`cancellation_rate` and `punctuality_rate` were first computed in steps 9–10 using uncorrected counts. They are recomputed here from the now-corrected source columns to ensure consistency.

In [140]:
df["cancellation_rate"] = df["Number of cancelled trains"] / df["Number of scheduled trains"]
df["punctuality_rate"]  = 1 - (df["Number of trains delayed at arrival"] / df["Number of scheduled trains"])

print(f'cancellation_rate — min: {df["cancellation_rate"].min():.4f}  mean: {df["cancellation_rate"].mean():.4f}  max: {df["cancellation_rate"].max():.4f}')
print(f'punctuality_rate  — min: {df["punctuality_rate"].min():.4f}  mean: {df["punctuality_rate"].mean():.4f}  max: {df["punctuality_rate"].max():.4f}')

cancellation_rate — min: 0.0000  mean: inf  max: inf
punctuality_rate  — min: 0.0000  mean: 0.8610  max: 1.0000


### 12. Export — cleaned dataset

Index is reset to 0-based after row removals, then the cleaned dataset is written to `../cleaned_dataset.csv`.

In [141]:
df = df.reset_index(drop=True)
df.to_csv("../cleaned_dataset.csv", index=False)
print(f'Exported: {len(df)} rows, {len(df.columns)} columns → ../cleaned_dataset.csv')
print(f'Columns: {df.columns.tolist()}')

Exported: 11260 rows, 29 columns → ../cleaned_dataset.csv
Columns: ['Date', 'Service', 'Departure station', 'Arrival station', 'Average journey time', 'Number of scheduled trains', 'Number of cancelled trains', 'Number of trains delayed at departure', 'Average delay of late trains at departure', 'Average delay of all trains at departure', 'Number of trains delayed at arrival', 'Average delay of late trains at arrival', 'Average delay of all trains at arrival', 'Number of trains delayed > 15min', 'Average delay of trains > 15min (if competing with flights)', 'Number of trains delayed > 30min', 'Number of trains delayed > 60min', 'Pct delay due to external causes', 'Pct delay due to infrastructure', 'Pct delay due to traffic management', 'Pct delay due to rolling stock', 'Pct delay due to station management and equipment reuse', 'Pct delay due to passenger handling (crowding, disabled persons, connections)', 'year', 'month', 'season', 'delay_category', 'cancellation_rate', 'punctuality_r

### 13. Data quality audit — model readiness

Self-evaluation metrics that quantify the cleaned dataset's readiness for downstream modelling. All metrics are appended to the report accumulator and exported to `cleaning_report.csv`.

| Sub-section | What it measures |
|-------------|-----------------|
| 13.1 | Completeness — fraction of non-null cells overall and per column |
| 13.2 | Numeric distributions — n, mean, std, quartiles, min/max |
| 13.3 | Outlier rates — fraction of values with \|z-score\| > 3 |
| 13.4 | Validity checks — binary invariants that must hold after cleaning |
| 13.5 | Pipeline effectiveness — total corrections and algebraic recoveries |

#### 13.1 Completeness

Overall fraction of non-null cells plus per-column breakdown. The five least and most complete columns are printed for a quick diagnosis of residual sparsity.

In [142]:
_total_cells  = df.shape[0] * df.shape[1]
_null_cells   = int(df.isnull().sum().sum())
_completeness = round(1 - _null_cells / _total_cells, 4)

_col_completeness = (1 - df.isnull().mean()).sort_values()

report.append({'metric': 'overall_completeness', 'value': _completeness,
               'category': 'quality_score', 'reason': 'Non-null cells / total cells (1.0 = fully complete)'})
for col in df.columns:
    rate = round(float(1 - df[col].isnull().mean()), 4)
    report.append({'metric': f'completeness__{col}', 'value': rate,
                   'category': 'completeness_per_column', 'reason': col})

print(f'Overall completeness : {_completeness:.2%}  ({_total_cells - _null_cells:,} / {_total_cells:,} cells non-null)')
print(f'\nLeast complete columns:')
for col, rate in _col_completeness.head(5).items():
    bar = '░' * int((1 - rate) * 30)
    print(f'  {rate:>6.1%}  {bar}  {col}')
print(f'\nMost complete columns:')
for col, rate in _col_completeness.tail(5).items():
    print(f'  {rate:>6.1%}  {col}')

Overall completeness : 98.72%  (322,345 / 326,540 cells non-null)

Least complete columns:
   97.4%    punctuality_rate
   97.4%    Pct delay due to station management and equipment reuse
   97.5%    Average journey time
   97.5%    Pct delay due to external causes
   97.5%    Pct delay due to traffic management

Most complete columns:
  100.0%  Service
  100.0%  year
  100.0%  month
  100.0%  season
  100.0%  Date


#### 13.2 Numeric distribution statistics

Descriptive stats (n, mean, std, quartiles, min/max) for key numeric columns. Used to verify that cleaning did not distort natural distributions and to detect residual skew or scale anomalies.

In [143]:
_dist_cols = [
    "Number of scheduled trains",
    "Average delay of all trains at arrival",
    "Average delay of all trains at departure",
    "Average delay of late trains at arrival",
    "Average delay of late trains at departure",
    "cancellation_rate",
    "punctuality_rate",
    "Average journey time",
    "Pct delay due to external causes",
    "Pct delay due to infrastructure",
    "Pct delay due to traffic management",
    "Pct delay due to rolling stock",
]

_stat_rows = []
for col in _dist_cols:
    if col not in df.columns:
        continue
    s = df[col].dropna()
    if s.empty:
        continue
    row = {
        'column': col, 'n': len(s),
        'mean':   round(float(s.mean()), 3),
        'std':    round(float(s.std()),  3),
        'min':    round(float(s.min()),  3),
        'p25':    round(float(s.quantile(0.25)), 3),
        'median': round(float(s.median()), 3),
        'p75':    round(float(s.quantile(0.75)), 3),
        'max':    round(float(s.max()),  3),
    }
    _stat_rows.append(row)
    for stat, val in row.items():
        if stat == 'column':
            continue
        report.append({'metric': f'dist__{col}__{stat}', 'value': val,
                       'category': 'numeric_distribution', 'reason': col})

_stats_df = pd.DataFrame(_stat_rows).set_index('column')
print(_stats_df.to_string())

                                               n     mean      std      min      p25   median      p75       max
column                                                                                                          
Number of scheduled trains                 11260  270.671  182.872    0.000  150.000  229.000  358.000  1100.000
Average delay of all trains at arrival     11251    6.078    5.465 -173.077    3.359    5.332    8.075    92.000
Average delay of all trains at departure   11250    3.149    5.189 -229.269    1.208    2.323    3.952   115.047
Average delay of late trains at arrival    11242   35.499   15.986  -40.371   25.884   33.739   42.853   299.600
Average delay of late trains at departure  11250   12.319   11.953   -5.370    6.174   10.348   15.775   316.188
cancellation_rate                          11006      inf      NaN    0.000    0.000    0.007    0.031       inf
punctuality_rate                           10964    0.861    0.077    0.000    0.826    0.874   

/home/sacha-lma/Project/Epitech_Projects/Tek_1/AIA/Tardis/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


#### 13.3 Outlier rates

For each numeric column: fraction of rows with absolute z-score > 3. High rates signal columns with extreme values that survived cleaning and may need further treatment before modelling.

In [144]:
_outlier_cols = [c for c in df.select_dtypes(include='number').columns
                 if c not in ('year', 'month')]
_outlier_rows = []

for col in _outlier_cols:
    s = df[col].dropna()
    if len(s) < 10 or s.std() == 0:
        continue
    z     = (s - s.mean()) / s.std()
    n_out = int((z.abs() > 3).sum())
    rate  = round(n_out / len(df), 4)
    _outlier_rows.append((col, n_out, rate))
    report.append({
        'metric':   f'outlier_rate__{col}',
        'value':    rate,
        'category': 'outlier_analysis',
        'reason':   f'{n_out} values |z|>3 out of {len(df)} rows',
    })

_outlier_rows.sort(key=lambda x: -x[2])
print(f'{"Column":<58} {"N outliers":>10}  {"Rate":>7}')
print('─' * 82)
for col, n, rate in _outlier_rows:
    bar = '█' * min(int(rate * 300), 30)
    print(f'{col:<58} {n:>10,}  {rate:>6.2%}  {bar}')

Column                                                     N outliers     Rate
──────────────────────────────────────────────────────────────────────────────────
Number of cancelled trains                                        244   2.17%  ██████
Number of trains delayed > 60min                                  218   1.94%  █████
Number of trains delayed > 30min                                  200   1.78%  █████
Pct delay due to passenger handling (crowding, disabled persons, connections)        190   1.69%  █████
Number of trains delayed at departure                             186   1.65%  ████
Number of trains delayed > 15min                                  178   1.58%  ████
Number of trains delayed at arrival                               164   1.46%  ████
Pct delay due to infrastructure                                   148   1.31%  ███
Pct delay due to station management and equipment reuse           144   1.28%  ███
punctuality_rate                                            

/home/sacha-lma/Project/Epitech_Projects/Tek_1/AIA/Tardis/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


#### 13.4 Feature validity checks

Binary assertions on invariants that must hold after cleaning — each outputs `1` (pass) or `0` (fail). Any failure signals a residual data quality issue that the pipeline did not correct.

In [145]:
_checks = {}

_checks['cancellation_rate_in_0_1']  = int(df['cancellation_rate'].dropna().between(0, 1).all())
_checks['punctuality_rate_in_0_1']   = int(df['punctuality_rate'].dropna().between(0, 1).all())
_checks['no_negative_scheduled']     = int((df['Number of scheduled trains'].dropna() >= 0).all())

_hier_pairs = [
    ('Number of trains delayed > 15min', 'Number of trains delayed > 30min'),
    ('Number of trains delayed > 30min', 'Number of trains delayed > 60min'),
]
for col_lo, col_hi in _hier_pairs:
    both = df[[col_lo, col_hi]].dropna()
    key  = f'hierarchy_{col_lo.split(">")[1].strip().replace(" ", "")}_{col_hi.split(">")[1].strip().replace(" ", "")}'
    _checks[key] = int((both.iloc[:, 0] >= both.iloc[:, 1]).all())

_pct_cols = [c for c in df.columns if c.startswith('Pct delay due to')]
if _pct_cols:
    _row_pct_sum = df[_pct_cols].sum(axis=1, skipna=True)
    _checks['pct_delay_sum_le_105'] = int((_row_pct_sum.dropna() <= 105).all())

_checks['date_range_sane']         = int((df['Date'] >= '2015-01-01').all() and (df['Date'] <= '2026-12-31').all())
_checks['service_only_2_values']   = int(df['Service'].dropna().isin(['NATIONAL', 'INTERNATIONAL']).all())
_checks['no_fully_null_rows']      = int(df.isnull().all(axis=1).sum() == 0)

_all_passed = sum(_checks.values())
for k, v in _checks.items():
    report.append({'metric': f'validity__{k}', 'value': v, 'category': 'validity_checks', 'reason': '1=pass  0=fail'})
report += [
    {'metric': 'validity_checks_passed', 'value': _all_passed,      'category': 'validity_checks', 'reason': f'{_all_passed}/{len(_checks)} checks passed'},
    {'metric': 'validity_checks_total',  'value': len(_checks),     'category': 'validity_checks', 'reason': 'Total validity rules evaluated'},
]

print('Validity checks:')
for k, v in _checks.items():
    print(f'  {"✓" if v else "✗"} {k}')
print(f'\n{_all_passed}/{len(_checks)} checks passed')

Validity checks:
  ✗ cancellation_rate_in_0_1
  ✓ punctuality_rate_in_0_1
  ✓ no_negative_scheduled
  ✗ hierarchy_15min_30min
  ✗ hierarchy_30min_60min
  ✓ pct_delay_sum_le_105
  ✓ date_range_sane
  ✓ service_only_2_values
  ✓ no_fully_null_rows

6/9 checks passed


#### 13.5 Pipeline effectiveness

Composite metrics quantifying how much work the pipeline performed: **corrections** (impossible values fixed) and **recoveries** (missing values derived algebraically). The correction rate per cell measures overall data noise density.

In [146]:
_total_corrections = sum(
    next(r['value'] for r in report if r['metric'] == m)
    for m in ['values_fixed_negative', 'values_fixed_count_vs_scheduled', 'values_fixed_delay_hierarchy']
)
_total_recovered = sum(
    next(r['value'] for r in report if r['metric'] == m)
    for m in ['values_filled_dep_late', 'values_filled_dep_all', 'values_filled_arr_late', 'values_filled_arr_all']
)
_total_cells_final   = df.shape[0] * df.shape[1]
_correction_rate     = round(_total_corrections / _total_cells_final, 6)
_initial_null_lookup = next(r['value'] for r in report if r['metric'] == 'initial_null_total')
_recovery_vs_null    = round(_total_recovered / max(_initial_null_lookup, 1), 4)

report += [
    {'metric': 'total_corrections',        'value': _total_corrections,  'category': 'pipeline_effectiveness', 'reason': 'Negatives + count overflows + hierarchy violations corrected'},
    {'metric': 'total_recovered_cells',    'value': _total_recovered,    'category': 'pipeline_effectiveness', 'reason': 'Cells recovered by algebraic derivation (all 4 delay columns)'},
    {'metric': 'correction_rate_per_cell', 'value': _correction_rate,    'category': 'pipeline_effectiveness', 'reason': 'Corrections / total final cells — data noise density'},
    {'metric': 'recovery_vs_initial_null', 'value': _recovery_vs_null,   'category': 'pipeline_effectiveness', 'reason': 'Recovered / initial null count — algebraic coverage'},
]
print(f'Total corrections : {_total_corrections:,}')
print(f'Total recovered   : {_total_recovered:,}')
print(f'Correction rate   : {_correction_rate:.4%} of all cells')
print(f'Recovery vs nulls : {_recovery_vs_null:.1%} of initial nulls recovered algebraically')

Total corrections : 427
Total recovered   : 1,080
Correction rate   : 0.1308% of all cells
Recovery vs nulls : 2.8% of initial nulls recovered algebraically


### 14. Export — cleaning audit report

The final dataset shape and null counts are appended to the report accumulator, which is then written to `../cleaning_report.csv`. This file provides a full audit trail of every transformation applied during the pipeline, with one entry per metric.

In [147]:
_remaining_null = int(df.isnull().sum().sum())
report += [
    {'metric': 'final_rows',         'value': len(df),                              'category': 'final_state', 'reason': 'Rows after full cleaning pipeline'},
    {'metric': 'final_columns',      'value': len(df.columns),                      'category': 'final_state', 'reason': 'Columns after cleaning + feature engineering'},
    {'metric': 'final_null_total',   'value': _remaining_null,                      'category': 'final_state', 'reason': 'Intentionally kept null — non-derivable columns'},
    {'metric': 'total_rows_removed', 'value': original_file.shape[0] - len(df),     'category': 'final_state', 'reason': 'Total rows removed across all cleaning steps'},
]

report_df = pd.DataFrame(report, columns=['metric', 'value', 'category', 'reason'])
report_df.to_csv("../cleaning_report.csv", index=False)

print(f'Report saved: {len(report_df)} entries → ../cleaning_report.csv')
print(report_df['category'].value_counts().to_string())

Report saved: 224 entries → ../cleaning_report.csv
category
numeric_distribution       96
completeness_per_column    29
initial_null_per_column    26
outlier_analysis           21
validity_checks            11
initial_state               7
cleaning                    6
station_clustering          5
data_recovery               4
pipeline_summary            4
pipeline_effectiveness      4
final_state                 4
recovery_summary            3
data_correction             3
quality_score               1
